# English resumes and job descriptions: a first look

This notebook answers simple questions: How many records do we have? What does one resume look like? Which jobs occur most often? What information is missing?

**No synthetic records are added.** The resumes are published LiveCareer examples collected in a public dataset; we have not verified that each describes a real person. The jobs are historical postings from 2023–2024, not a list of currently open vacancies.

The reusable loading code lives in `data-acquisition/scripts/authentic/load_data.py`. This notebook imports it so the script and notebook use the same cleaning rules. Charts and exploration stay here.

In [ ]:
from pathlib import Path
import os
import sys

search_roots = [Path.cwd(), *Path.cwd().parents]
ACQUISITION_ROOT = next((
    candidate
    for base in search_roots
    for candidate in (base, base / "data-acquisition")
    if (candidate / "scripts/authentic/load_data.py").exists()
), None)
if ACQUISITION_ROOT is None:
    raise RuntimeError("Open this notebook from inside the marketplace-ranking-system project.")
PROJECT_ROOT = ACQUISITION_ROOT.parent
sys.path.insert(0, str(ACQUISITION_ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(PROJECT_ROOT / ".cache/matplotlib"))

import json
import re
import textwrap
from zipfile import ZipFile
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymupdf
from IPython.display import display, Image, Markdown
from scripts.authentic.load_data import authentic_data_directory, load_datasets

DATA = authentic_data_directory()
REPORTS = DATA / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_colwidth", 95)
print("Python:", sys.executable)
print("Data folder:", DATA)

## Load the data

Run `.venv/bin/python data-acquisition/scripts/authentic/load_data.py` once before opening this notebook. If the cleaned files are missing, this cell rebuilds them using downloaded archives, without network access.

The loader removes empty text, repeated IDs, exact repeated text, and records that do not pass the English filter. Original records remain in the ZIP archives. English detection is an automatic estimate, not a manual language review.

In [ ]:
required = ["resumes.parquet", "jobs.parquet", "resumes_audit.parquet", "jobs_audit.parquet", "manifest.json"]
if not all((DATA / "processed" / name).exists() for name in required):
    load_datasets(DATA, offline=True)
resumes = pd.read_parquet(DATA / "processed/resumes.parquet")
jobs = pd.read_parquet(DATA / "processed/jobs.parquet")
audits = {name: pd.read_parquet(DATA / f"processed/{name}_audit.parquet") for name in ["resumes", "jobs"]}
manifest = json.loads((DATA / "processed/manifest.json").read_text())
summary = pd.DataFrame({name: item["counts"] for name, item in manifest["sources"].items()}).fillna(0).astype(int)
display(summary)
assert len(resumes) == summary.loc["accepted", "resumes"]
assert len(jobs) == summary.loc["accepted", "jobs"]

## Where did these records come from?

A resume row is one resume example, not a verified unique person. A job row is one posting. Two different job postings may use the same description, so the cleaned table keeps only the first exact text copy. Slightly rewritten copies may still remain.

In [ ]:
display(pd.DataFrame([
    {"dataset": name, "source": item["url"], "raw_rows": item["counts"]["raw_rows"],
     "kept_rows": item["counts"].get("accepted", 0), "archive_MB": round(item["archive_bytes"] / 1e6, 1)}
    for name, item in manifest["sources"].items()
]))
fig, ax = plt.subplots(figsize=(9, 3.5))
summary.loc[["raw_rows", "accepted"]].T.plot.bar(ax=ax, rot=0, color=["#b9c6d1", "#2a9d8f"])
ax.set(title="Records downloaded and records kept", ylabel="Number of records", xlabel="")
ax.legend(["Downloaded", "English, nonempty, deduplicated"])
plt.tight_layout()
fig.savefig(REPORTS / "record_counts.png", dpi=150)
plt.show()

## Read an actual resume and an actual job

These are text excerpts, not generated summaries. Basic email/phone masking is applied to displayed excerpts; it is not complete anonymization. Source category labels such as `INFORMATION-TECHNOLOGY` are broad categories, not extracted job titles.

In [ ]:
def preview(text, length=1100):
    text = re.sub(r"[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}", "[email removed]", str(text))
    text = re.sub(r"(?<!\w)(?:\+?\d[\d ().-]{8,}\d)(?!\w)", "[number removed]", text)
    return textwrap.fill(text[:length], width=105)

pool = resumes.loc[resumes["category"].eq("INFORMATION-TECHNOLOGY")]
resume = (pool if len(pool) else resumes).sample(1, random_state=42).iloc[0]
pool = jobs.loc[jobs["title"].str.contains("software engineer", case=False, na=False)]
job = (pool if len(pool) else jobs).sample(1, random_state=42).iloc[0]
print("RESUME CATEGORY:", resume["category"])
print(preview(resume["resume_text"]))
print("\nJOB:", job["title"], "|", job["company_name"], "|", job["location"])
print(preview(job["description"]))

## See the original resume layout

This is the first page of the same resume shown above, read directly from the downloaded ZIP. It is not a newly generated resume. The raw archive also contains the other original PDFs.

In [ ]:
with ZipFile(DATA / "raw/resumes/resume-dataset.zip") as archive:
    pdf_bytes = archive.read(resume["pdf_archive_member"])
with pymupdf.open(stream=pdf_bytes, filetype="pdf") as document:
    print("Pages in this resume:", len(document))
    page_image = document[0].get_pixmap(dpi=90).tobytes("png")
display(Image(data=page_image, width=700))

## What columns do we have?

For resumes we have text and a category. We do **not** yet have trustworthy structured years of experience, education, or technical skills. Those need a later extraction step. Job postings already include title, company, location, work type, and some experience-level labels.

In [ ]:
def column_info(frame):
    missing = frame.isna() | frame.astype("string").apply(lambda s: s.str.strip().eq(""))
    return pd.DataFrame({"type": frame.dtypes.astype(str), "missing_percent": (100 * missing.mean()).round(1)})
print("Resume columns")
display(column_info(resumes))
print("Selected job columns")
job_columns = ["title", "description", "company_name", "location", "formatted_work_type",
               "formatted_experience_level", "source_skill_tags", "min_salary", "max_salary", "currency"]
display(column_info(jobs[job_columns]))

## Which resume categories and job titles are most common?

If some categories dominate, an overall matching score may mostly describe those categories. We should later evaluate matching by occupation as well.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
resumes["category"].value_counts().sort_values().plot.barh(ax=axes[0], color="#2a9d8f")
axes[0].set(title="Resume examples by source category", xlabel="Resumes", ylabel="")
jobs["title"].value_counts().head(12).sort_values().plot.barh(ax=axes[1], color="#457b9d")
axes[1].set(title="12 most common exact job titles", xlabel="Job descriptions", ylabel="")
plt.tight_layout()
fig.savefig(REPORTS / "categories_and_titles.png", dpi=150)
plt.show()

## How long are the documents?

Very short descriptions may lack requirements. Very long resumes may contain repeated sections. Length alone does not prove that a record is good or bad. The charts stop at the 99th percentile for readability; the statistics include all records.

In [ ]:
resume_words = resumes["resume_text"].str.split().str.len()
job_words = jobs["description"].str.split().str.len()
display(pd.DataFrame({"resume_words": resume_words.describe(percentiles=[.5, .9, .99]),
                      "job_words": job_words.describe(percentiles=[.5, .9, .99])}).round(0))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, values, title in zip(axes, [resume_words, job_words], ["Resume length", "Job description length"]):
    sns.histplot(values[values <= values.quantile(.99)], bins=40, ax=ax)
    ax.set(title=title + " (up to 99th percentile)", xlabel="Words", ylabel="Records")
plt.tight_layout()
fig.savefig(REPORTS / "document_lengths.png", dpi=150)
plt.show()

## What job information is missing?

For example, an empty experience-level field means the dataset did not provide a level. It does not mean the job requires no experience. Missing salary values are not zero salaries.

In [ ]:
missing = column_info(jobs[job_columns])["missing_percent"].sort_values()
fig, ax = plt.subplots(figsize=(10, 4))
missing.plot.barh(ax=ax, color="#e9c46a")
ax.set(title="Missing fields in accepted job postings", xlabel="Percent missing", ylabel="", xlim=(0, 100))
plt.tight_layout()
fig.savefig(REPORTS / "job_missing_fields.png", dpi=150)
plt.show()
display(jobs["formatted_work_type"].fillna("Unknown").value_counts().rename("jobs").to_frame())
display(jobs["formatted_experience_level"].fillna("Unknown").value_counts().rename("jobs").to_frame())
print("Original posting dates:", jobs["original_listed_time_utc"].min(), "to", jobs["original_listed_time_utc"].max())

## Simple skill-word counts

This counts mentions such as Python and Excel. It is a small illustration, **not a skill extractor**: a mention does not prove ability, and a job may mention a skill as optional. The supplied `source_skill_tags` are mostly broad job functions, not a detailed technical-skill list.

In [ ]:
patterns = {"Python": r"\bpython\b", "SQL": r"\bsql\b", "Excel": r"\bexcel\b",
            "Java": r"\bjava\b", "JavaScript": r"\bjavascript\b", "AWS": r"\baws\b",
            "Project management": r"\bproject management\b", "Customer service": r"\bcustomer service\b"}
mentions = pd.DataFrame({
    label: {"Resumes": 100 * resumes["resume_text"].str.contains(pattern, case=False, na=False).mean(),
            "Jobs": 100 * jobs["description"].str.contains(pattern, case=False, na=False).mean()}
    for label, pattern in patterns.items()
}).T
fig, ax = plt.subplots(figsize=(11, 4))
mentions.plot.bar(ax=ax, rot=30)
ax.set(title="Records mentioning selected words", ylabel="Percent of records", xlabel="")
plt.tight_layout()
fig.savefig(REPORTS / "skill_word_mentions.png", dpi=150)
plt.show()
display(mentions.round(1))

## Inspect the English filter and duplicate handling

The audit tables keep one row per input record, including excluded records. They explain what was removed without deleting anything from the original downloads. Exact text deduplication does not detect all near-duplicates or establish unique people.

In [ ]:
for name, audit in audits.items():
    print(name.upper())
    display(audit["status"].value_counts().rename("records").to_frame())
    display(audit["language"].value_counts().head(8).rename("detected_records").to_frame())
    rejected = audit.loc[~audit["status"].eq("accepted")]
    if len(rejected):
        display(rejected.head(5))
for frame, identity, text in [(resumes, "resume_id", "resume_text"), (jobs, "job_id", "description")]:
    assert frame[identity].is_unique
    assert frame["text_sha256"].is_unique
    assert frame["language"].eq("en").all()
    assert frame["language_confidence"].ge(manifest["english_min_confidence"]).all()
    assert frame[text].str.len().gt(0).all()
print("Basic data checks passed.")

## What we learned, and what comes next

The summary below is computed from the loaded files. The next task would be extracting skills and experience from resume text, with evidence from the original wording. This notebook does not invent candidate facts or create matching labels.

In [ ]:
display(Markdown(
    f"- **{len(resumes):,} English resume examples** kept from {len(audits['resumes']):,} downloaded rows.\n"
    f"- **{len(jobs):,} English job descriptions** kept from {len(audits['jobs']):,} downloaded rows.\n"
    f"- **{resumes['category'].nunique()} resume categories**; the largest is **{resumes['category'].value_counts().index[0]}**.\n"
    f"- Median length: **{resume_words.median():,.0f} words per resume**, **{job_words.median():,.0f} words per job**.\n"
    f"- **{jobs['formatted_experience_level'].isna().mean():.1%}** of accepted jobs have no supplied experience-level label.\n"
    "- Resume categories and job-function tags are not detailed skill profiles.\n"
    "- These sources have different dates and coverage; they are not a linked hiring-outcome dataset.\n"
    "- Original archives stay in `data/authentic/raw`; reusable cleaned tables stay in `data/authentic/processed`."
))